# Checks & Debugs regarding refactoring of code
Felix Zaussinger | XX.YY.ZZZZ

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")

In [2]:
from src.data.framework import Esco, Onet
esco = Esco()
onet = Onet()

ESCO Class

In [3]:
esco.occupations
esco.skills

,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage staff of music\ncoordinate duties of mu...,NaN,released,2016-12-20T17:43:43Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,oversee prison procedures\nmanage correctional...,NaN,released,2016-12-20T20:17:49Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,NaN,released,2016-12-20T19:18:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,monitoring of compliance with railway vehicles...,NaN,released,2016-12-20T20:02:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,establish available services\ndetermine rehabi...,NaN,released,2016-12-20T20:15:17Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Identify the different services available for ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13886,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/ffef5eb3-a15e...,skill/competence,sector-specific,remediate healthcare user's occupational perfo...,restore healthcare user's occupational perform...,NaN,released,2016-12-20T19:25:53Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,"Remediate or restore the cognitive, sensorimot..."
13887,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0b074-5a76...,skill/competence,sector-specific,install transport equipment lighting,install transport equipment illumination\nfix ...,NaN,released,2016-12-20T20:03:21Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Install lighting elements in transport equipme...
13888,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0e2cd-d0bd...,knowledge,sector-specific,natural language processing,NLP,NaN,released,2016-08-04T15:19:37Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,The technologies which enable ICT devices to u...
13889,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff5bc45-b506...,skill/competence,cross-sector,coordinate construction activities,reviewing construction progress\nconstruction ...,NaN,released,2016-12-20T18:22:35Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Coordinate the activities of several construct...


Onet Class

In [4]:
onet.green_occupations_gtp
onet.green_occupations_vona2018
onet.green_tasks_gtp
onet.brown_occupations_vona2018
onet.green_occupations_narrow_jrc

,isco08_jrc,occ_eng_jrc,n_green_tasks_jrc,n_tasks_jrc,greenness_jrc
0,1.1.2.4.3,General managers and equivalent in health care,1,11.0,0.090909
1,1.1.2.6.3,Managers and equivalent in health care,1,13.0,0.076923
2,1.2.2.3.0,Directors and general managers of construction...,1,13.0,0.076923
3,1.2.3.2.0,"Directors and managers of the organisation, hu...",1,15.0,0.066667
4,1.2.3.7.0,Directors and managers of the research and dev...,1,12.0,0.083333
5,1.2.3.9.0,Other departmental directors and managers,1,10.0,0.100000
6,1.3.1.2.0,Entrepreneurs and managers of small companies ...,1,14.0,0.071429
7,1.3.1.3.0,Entrepreneurs and managers of small constructi...,1,14.0,0.071429
8,1.3.1.8.0,Entrepreneurs and managers of small companies ...,1,13.0,0.076923
9,2.1.1.2.1,Chemists and related professions,1,14.0,0.071429


Calculating occ-skills matrix with more flexibility

In [5]:
osm_weighted = esco.read_occ_skills_matrix(return_version="weighted")
np.unique(osm_weighted)

array([0. , 0.5, 1. ])

In [6]:
osm_unweighted = esco.read_occ_skills_matrix(return_version="unweighted", assign_labels=True)
np.unique(osm_unweighted)

array([0, 1], dtype=int64)

In [7]:
osm_encoded = esco.read_occ_skills_matrix(return_version="raw", assign_labels=True)
np.unique(osm_encoded)

array([0, 1, 2], dtype=int64)

In [8]:
osm_weighted = esco.label_osm(osm_weighted)
osm_weighted

,manage musical staff,supervise correctional procedures,apply anti-oppressive practices,control compliance of railway vehicles regulations,identify available services,perform toxicological studies,ensure coquille uniformity,Haskell,show initiative,train staff to reduce food waste,apply diplomatic principles,lead police investigations,handle fish harvesting waste,develop energy saving concepts,perform street interventions in social work,work with soloists,sport and exercise medicine,conduct research on flora,install heat pump,design biomass installations,handle equipment while suspended,teach housekeeping skills,check train engines,influence public policies,enterprise risk management,manufacture ingredients,maintain aquaculture ponds,apply credit risk policy,handle customer requests related to cargo,draft scientific or academic papers and technical documentation,Incremental development,use of special equipment for daily activities,sawing techniques,produce guitar components,operate agricultural machinery,control pyrotechnics stock,guarantee customer satisfaction,manufacture wearing apparel products,cure tobacco leaves,develop a rehabilitation programme,maintain inventory of cleaning supplies,cold vulcanisation,supervise housekeeping operations,act as contact person during equipment incident,manage time in landscaping,advise on customs regulations,manage university department,develop terminology databases,assess nutritional characteristics of food,repair electric bicycles,maintain sorting equipment,explain features in accommodation venue,types of barley,KDevelop,purchase vehicle parts,inspect offshore constructions,transport patient to medical facility,adjust envelope cutting settings,prepare oils,purchase supplies,...,establish gaming policies,media and information literacy,evaluate clinical outcomes of dental hygiene interventions,gem cutting forms,cut pig's teeth,provide therapy of the visual system,administer materials to tea bag machines,advise on bank account,manage dental emergencies,assume highest level of responsibility in inland water transportation,manage time in food processing operations,student financial aid programmes,interpret pedigree charts,perform software recovery testing,astronomy,use instruments for food measurement,tend ball mill,understand spoken Norwegian,cut filament,procure time sheet approval,communicate with a non-scientific audience,prepare soda-ash,microchip scanners,rope lashing,create social alliances,assess your competencies in leading community arts,write Occitan,arrange customs inspection,perform nutrition analysis,analyse work-related written reports,surveillance methods,remove old caulking,keep company,clean patients' ear canals,anodising process,preserve samples,treat vehicle fabrics,collect domestic waste,perform energy simulations,match vessels to shipping routes,types of drill bits,cable-propelled transit,negotiate rights of use,Capture One,precious metal processing,control train movement,dependency on drugs,organise vehicle parts storage,set up reinforcing steel,model sensor,Scala,operate forestry equipment,test soil load bearing capacity,buy new library items,design clocks,remediate healthcare user's occupational performance,install transport equipment lighting,natural language processing,coordinate construction activities,position guardrails and toeboards
technical director,0.0,0,0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
metal drawing machine operator,0.0,0,0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0

In [9]:
osm_weighted[osm_weighted == 1].sum(axis=1).describe()

count    3008.000000
mean       21.640957
std        12.145173
min         0.000000
25%        14.000000
50%        19.000000
75%        26.000000
max       133.000000
dtype: float64

In [10]:
osm_weighted[osm_weighted == 0.5].sum(axis=1).describe()

count    3008.000000
mean        9.766622
std        10.787132
min         0.000000
25%         4.500000
50%         8.500000
75%        11.500000
max       159.500000
dtype: float64

In [11]:
osm_unweighted[osm_unweighted == 1].sum(axis=1).describe()

count    3008.000000
mean       41.174202
std        24.990325
min         4.000000
25%        27.000000
50%        36.000000
75%        48.000000
max       340.000000
dtype: float64

Combining skills metadata

In [12]:
smd = esco.combine_skills_metadata(variable_selection=None)
smd

C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description,broaderConceptUri,broaderConceptPT,alternative_class_eth,skill_green_eth,skill_green_esco,impact_eth,scope_eth,machineScore_green,comment_eth,skill_brown_eth,alternative_class_eth_brown,scope_eth_brown,impact_eth_brown,comment_eth_brown,skill_brown_esco,machineScore_brown,skill_neutral_esco,skill_classification_esco,coreness
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage staff of music\ncoordinate duties of mu...,NaN,released,2016-12-20T17:43:43Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.002187
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,oversee prison procedures\nmanage correctional...,NaN,released,2016-12-20T20:17:49Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Supervise the operations of a correctional fac...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.000000
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,NaN,released,2016-12-20T19:18:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c...",NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.000748
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,monitoring of compliance with railway vehicles...,NaN,released,2016-12-20T20:02:19Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ...",NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.049473
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,establish available services\ndetermine rehabi...,NaN,released,2016-12-20T20:15:17Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Identify the different services available for ...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.001577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13886,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/ffef5eb3-a15e...,skill/competence,sector-specific,remediate healthcare user's occupational perfo...,restore healthcare user's occupational perform...,NaN,released,2016-12-20T19:25:53Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,"Remediate or restore the cognitive, sensorimot...",NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.000103
13887,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0b074-5a76...,skill/competence,sector-specific,install transport equipment lighting,install transport equipment illumination\nfix ...,NaN,released,2016-12-20T20:03:21Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Install lighting elements in transport equipme...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.013943
13888,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/fff0e2cd-d0bd...,knowledge,sector-specific,natural language processing,NLP,NaN,released,2016-08-04T15:19:37Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,The technologies which enable ICT devices to u...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,True,neutral,0.025162
13889,Knowledg

Calculating GBN shares at occupation level

In [13]:
esco.calc_gbn_shares_skill_based(skills_metadata=smd)

,conceptUri,n_total_specific_skills,n_green_specific_skills_esco,share_green_esco,n_brown_specific_skills_esco,share_brown_esco,n_neutral_specific_skills_esco,share_neutral_esco,gbn_classification_esco
0,http://data.europa.eu/esco/occupation/00030d09...,8,0,0.000000,0,0.000000,8,1.000000,neutral
1,http://data.europa.eu/esco/occupation/000e93a3...,39,0,0.000000,5,0.128205,34,0.871795,neutral
2,http://data.europa.eu/esco/occupation/0019b951...,43,0,0.000000,0,0.000000,43,1.000000,neutral
3,http://data.europa.eu/esco/occupation/0022f466...,39,2,0.051282,0,0.000000,37,0.948718,neutral
4,http://data.europa.eu/esco/occupation/002da35b...,26,0,0.000000,0,0.000000,26,1.000000,neutral
...,...,...,...,...,...,...,...,...,...
3003,http://data.europa.eu/esco/occupation/ff656b3a...,58,0,0.000000,0,0.000000,58,1.000000,neutral
3004,http://data.europa.eu/esco/occupation/ff8d4065...,26,14,0.538462,0,0.000000,12,0.461538,green
3005,http://data.europa.eu/esco/occupation/ffa4dd5d...,29,0,0.000000,1,0.034483,28,0.965517,neutral
3006,http://data.europa.eu/esco/occupation/ffade2f4...,31,0,0.000000,0,0.000000,31,1.000000,neutral


Preprocessing of EU-LFS data

In [28]:
config = utils.load_config(os.path.join(useful_paths.config_dir, "eu_lfs_config.yml"))
config

{'paths': {'raw': 'C:\\eurostat_data\\raw',
  'raw_yf': 'C:\\eurostat_data\\raw\\Yearly_Data\\YearlyFiles_83_2019\\_YearlyFiles',
  'interim': 'C:\\eurostat_data\\interim',
  'processed': 'C:\\eurostat_data\\processed',
  'fmt_folder': '{country}_YEAR_1998_onwards',
  'fmt_file': '{country}{year}_y.csv'},
 'preprocessing': {'n_digits_isco08': 3,
  'n_digits_nace': 1,
  'n_digits_nuts': 2,
  'years': ['2019'],
  'countries': ['AT',
   'BE',
   'BG',
   'CH',
   'CY',
   'CZ',
   'DE',
   'DK',
   'EE',
   'ES',
   'FI',
   'FR',
   'GR',
   'HR',
   'HU',
   'IE',
   'IS',
   'IT',
   'LT',
   'LU',
   'LV',
   'MT',
   'NL',
   'NO',
   'PL',
   'PT',
   'RO',
   'SE',
   'SI',
   'SK',
   'UK'],
  'countries_isco08_2d': ['BG', 'PL'],
  'countries_isco08_1d': ['MT'],
  'countries_nuts_1d': ['UK'],
  'countries_nuts_0d': ['MT', 'NL'],
  'scaling_factor_coeff': 1000,
  'pension_age': ['77', '82', '87', '92', '97'],
  'variables': ['WSTATOR',
   'SEX',
   'NACE1D',
   'ISCO3D',
   'COUNTR

In [47]:
from src.data.lfs import EuLfs

lfs = EuLfs(config=config)

In [51]:
lfs.read_filtered_file("DE", "2019", return_filtering_stats=False)

,SEX,WSTATOR,NACE1D,ISCO3D,COUNTRYW,REGIONW,FTPT,REFYEAR,COUNTRY,REGION,DEGURBA,HHTYPE,COEFF,AGE,ILOSTAT,EDUC4WN,HATLEV1D,MAINSTAT,INCDECIL,NUTS_ID,ISCO
0,1,1,C,214,DE,11,1,2019,DE,10,1,1,111.2350,52,1,0,H,1,10,DE11,214
1,2,1,F,311,DE,11,2,2019,DE,10,1,1,111.2350,42,1,0,H,1,02,DE11,311
8,1,1,C,432,DE,11,1,2019,DE,10,2,1,120.2525,32,1,0,L,1,09,DE11,432
11,1,1,H,831,DE,11,1,2019,DE,10,2,1,134.1850,42,1,0,M,1,09,DE11,831
12,2,1,G,432,DE,11,1,2019,DE,10,2,1,134.1850,37,1,0,M,1,09,DE11,432
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522043,2,1,Q,321,DE,G0,1,2019,DE,G0,2,1,141.3425,32,1,0,M,1,06,DEG0,321
522044,1,1,S,516,DE,G0,1,2019,DE,G0,2,1,141.3425,27,1,0,M,1,05,DEG0,516
522046,1,1,G,214,DE,G0,1,2019,DE,G0,3,1,133.3225,37,1,0,H,1,09,DEG0,214
522047,1,1,C,132,DE,G0,1,2019,DE,G0,3,1,130.2975,32,1,0,H,1,06,DEG0,132


In [57]:
lfs.preprocess_files(years=["2017"], countries=["AT", "DE"], save_file=True)

2017
AT
DE


In [60]:
lfs.read_preprocessed_file(year=2017)

,SEX,WSTATOR,NACE1D,ISCO3D,COUNTRYW,REGIONW,FTPT,REFYEAR,COUNTRY,REGION,DEGURBA,HHTYPE,COEFF,AGE,ILOSTAT,EDUC4WN,HATLEV1D,MAINSTAT,INCDECIL,NUTS_ID,ISCO
0,1,1,H,541,AT,22,1,2017,AT,20,2,1,35.6900,52,1,0,M,1,08,AT22,541
1,1,1,G,334,AT,22,1,2017,AT,20,2,1,63.8975,42,1,0,M,1,05,AT22,334
2,2,1,G,522,AT,22,2,2017,AT,20,2,1,52.1800,57,1,0,M,1,02,AT22,522
3,1,1,C,812,AT,22,1,2017,AT,20,2,1,57.9800,37,1,0,M,1,07,AT22,812
4,2,2,K,431,AT,22,1,2017,AT,20,2,1,57.9800,37,1,0,H,8,NaN,AT22,431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349968,2,2,P,341,DE,G0,2,2017,DE,G0,3,1,145.9850,27,1,0,H,1,07,DEG0,341
349969,1,1,C,721,DE,G0,1,2017,DE,G0,3,1,150.8525,27,1,0,M,1,07,DEG0,721
349970,2,1,G,522,DE,G0,1,2017,DE,G0,3,1,150.8525,27,1,0,M,1,05,DEG0,522
349971,1,1,M,216,DE,G0,1,2017,DE,G0,3,1,130.6400,37,1,0,H,1,08,DEG0,216


In [81]:
lfs.join_covariates(year=2017, isco_covariate_selection=["share_green_esco_mean"])

,AGE,COEFF,COEFF_share_green_esco_mean,COUNTRY,COUNTRYW,DEGURBA,EDUC4WN,FTPT,HATLEV1D,HHTYPE,ILOSTAT,INCDECIL,ISCO,ISCO3D,MAINSTAT,NACE1D,NACE1D_label,NUTS_ID,REFYEAR,REGION,REGIONW,SEX,WSTATOR,share_green_esco_mean
0,52,35.6900,1.303954,AT,AT,2,0,1,M,1,1,08,541,541,1,H,Transportation and Storage,AT22,2017,20,22,1,1,0.036536
1,42,63.8975,0.000000,AT,AT,2,0,1,M,1,1,05,334,334,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,AT22,2017,20,22,1,1,0.000000
2,57,52.1800,0.236335,AT,AT,2,0,2,M,1,1,02,522,522,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,AT22,2017,20,22,2,1,0.004529
3,37,57.9800,0.693583,AT,AT,2,0,1,M,1,1,07,812,812,1,C,Manufacturing,AT22,2017,20,22,1,1,0.011962
4,37,57.9800,0.000000,AT,AT,2,0,1,H,1,1,NaN,431,431,8,K,Financial and Insurance Activities,AT22,2017,20,22,2,2,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349968,27,145.9850,0.460371,DE,DE,3,0,2,H,1,1,07,341,341,1,P,Education,DEG0,2017,G0,G0,2,2,0.003154
349969,27,150.8525,1.742801,DE,DE,3,0,1,M,1,1,07,721,721,1,C,Manufacturing,DEG0,2017,G0,G0,1,1,0.011553
349970,27,150.8525,0.683244,DE,DE,3,0,1,M,1,1,05,522,522,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,DEG0,2017,G0,G0,2,1,0.004529
349971,37,130.6400,4.930682,DE,DE,3,0,1,H,1,1,08,216,216,1,M,"Professional, Scientific and Technical Activities",DEG0,2017,G0,G0,1,1,0.037743


In [82]:
lfs.read_merged_file(year=2017)

,AGE,COEFF,COEFF_share_green_esco_mean,COUNTRY,COUNTRYW,DEGURBA,EDUC4WN,FTPT,HATLEV1D,HHTYPE,ILOSTAT,INCDECIL,ISCO,ISCO3D,MAINSTAT,NACE1D,NACE1D_label,NUTS_ID,REFYEAR,REGION,REGIONW,SEX,WSTATOR,share_green_esco_mean
0,52,35.6900,1.303954,AT,AT,2,0,1,M,1,1,08,541,541,1,H,Transportation and Storage,AT22,2017,20,22,1,1,0.036536
1,42,63.8975,0.000000,AT,AT,2,0,1,M,1,1,05,334,334,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,AT22,2017,20,22,1,1,0.000000
2,57,52.1800,0.236335,AT,AT,2,0,2,M,1,1,02,522,522,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,AT22,2017,20,22,2,1,0.004529
3,37,57.9800,0.693583,AT,AT,2,0,1,M,1,1,07,812,812,1,C,Manufacturing,AT22,2017,20,22,1,1,0.011962
4,37,57.9800,0.000000,AT,AT,2,0,1,H,1,1,NaN,431,431,8,K,Financial and Insurance Activities,AT22,2017,20,22,2,2,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349968,27,145.9850,0.460371,DE,DE,3,0,2,H,1,1,07,341,341,1,P,Education,DEG0,2017,G0,G0,2,2,0.003154
349969,27,150.8525,1.742801,DE,DE,3,0,1,M,1,1,07,721,721,1,C,Manufacturing,DEG0,2017,G0,G0,1,1,0.011553
349970,27,150.8525,0.683244,DE,DE,3,0,1,M,1,1,05,522,522,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,DEG0,2017,G0,G0,2,1,0.004529
349971,37,130.6400,4.930682,DE,DE,3,0,1,H,1,1,08,216,216,1,M,"Professional, Scientific and Technical Activities",DEG0,2017,G0,G0,1,1,0.037743


In [83]:
df=lfs.read_merged_file(year=2019)

In [84]:
df

,AGE,COEFF,COEFF_share_brown_esco_mean,COEFF_share_green_esco_mean,COEFF_share_green_gtp_mean,COEFF_share_neutral_esco_mean,COUNTRY,COUNTRYW,DEGURBA,EDUC4WN,FTPT,HATLEV1D,HHTYPE,ILOSTAT,INCDECIL,ISCO,ISCO3D,MAINSTAT,NACE1D,NACE1D_label,NUTS_ID,REFYEAR,REGION,REGIONW,SEX,WSTATOR,isco_label_1,isco_label_2,isco_label_3,isco_label_4,isco_level_1,isco_level_2,isco_level_3,isco_level_4,n_occ_esco_sum,share_brown_esco_mean,share_green_esco_mean,share_green_gtp_mean,share_neutral_esco_mean
0,52,82.5525,0.998881,2.768554,21.297373,78.785065,AT,AT,3,0,1,M,1,1,NaN,132,132,1,C,Manufacturing,AT12,2019,10,12,1,1,NaN,NaN,"Manufacturing, mining, construction, and distr...",NaN,NaN,NaN,132,NaN,118.0,0.012100,0.033537,0.257986,0.954363
1,17,82.5525,1.117643,7.181130,32.827993,74.253727,AT,AT,3,1,1,L,1,1,NaN,311,311,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,AT12,2019,10,12,1,1,NaN,NaN,Physical and engineering science technicians,NaN,NaN,NaN,311,NaN,134.0,0.013539,0.086989,0.397662,0.899473
2,42,47.2325,0.000000,0.000000,0.000000,47.232500,AT,AT,3,0,1,M,1,1,NaN,411,411,1,G,Wholesale and Retail Trade; Repair of Motor Ve...,AT12,2019,10,12,2,1,NaN,NaN,General office clerks,NaN,NaN,NaN,411,NaN,2.0,0.000000,0.000000,0.000000,1.000000
3,47,52.9450,0.518645,2.668451,0.000000,49.757904,AT,AT,3,0,1,M,1,1,NaN,911,911,1,Q,Human Health and Social Work Activities,AT12,2019,10,12,2,1,NaN,NaN,"Domestic, hotel and office cleaners and helpers",NaN,NaN,NaN,911,NaN,7.0,0.009796,0.050400,0.000000,0.939804
4,47,75.8625,0.000000,0.225781,0.000000,75.636719,AT,AT,2,0,1,H,1,1,NaN,011,011,1,O,Public Administration and Defence; Compulsory ...,AT12,2019,10,12,1,1,NaN,NaN,Commissioned armed forces officers,NaN,NaN,NaN,011,NaN,12.0,0.000000,0.002976,0.000000,0.997024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1546564,17,1100.1400,0.000000,2.014908,0.000000,1098.125092,UK,UK,1,1,2,M,1,1,NaN,421,421,NaN,G,Wholesale and Retail Trade; Repair of Motor Ve...,UKM,2019,M0,M0,1,2,NaN,NaN,"Tellers, money collectors and related clerks",NaN,NaN,NaN,421,NaN,14.0,0.000000,0.001832,0.000000,0.998168
1546565,52,1790.6100,81.733828,175.305163,873.219677,1533.571009,UK,UK,3,0,1,H,1,1,10,211,211,NaN,O,Public Administration and Defence; Compulsory ...,UKM,2019,M0,M0,1,1,NaN,NaN,Physical and earth science professionals,NaN,NaN,NaN,211,NaN,29.0,0.045646,0.097902,0.487666,0.856452
1546566,57,1858.6800,40.006625,118.057101,419.701935,1700.616274,UK,UK,2,0,1,M,1,1,NaN,833,833,NaN,H,Transportation and Storage,UKM,2019,M0,M0,1,1,NaN,NaN,Heavy truck and bus drivers,NaN,NaN,NaN,833,NaN,11.0,0.021524,0.063517,0.225806,0.914959
1546567,57,1723.3000,0.000000,4.008418,0.000000,1719.291582,UK,UK,2,0,1,H,1,1,06,263,263,NaN,Q,Human Health and Social Work Activities,UKM,2019,M0,M0,2,1,NaN,NaN,Social and religious professionals,NaN,NaN,NaN,263,NaN,66.0,0.000000,0.002326,0.000000,0.997674
